In [ ]:
# ==============================================================
# Auto-fetch NYC TLC Yellow Taxi Trip Data (2024) + Zone Lookup
# --------------------------------------------------------------
# - Downloads 2024 monthly parquet files into ./yellow_2024/
# - Downloads taxi_zone_lookup.csv into the project root
# - Skips files that already exist (safe to re-run)
# - Ensures preprocessing pipeline has required raw inputs
# ==============================================================

import os
from pathlib import Path
from urllib.request import urlretrieve

# Base URLs for TLC trip data and zone lookup
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
ZONES_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# Months of interest for 2024
MONTHS_2024 = [
    "2024-01","2024-02","2024-03","2024-04","2024-05","2024-06",
    "2024-07","2024-08","2024-09","2024-10","2024-11","2024-12"
]

def download(url: str, dest: Path):
    """
    Download a file from the given URL to the specified destination path.
    Skips download if the file already exists.
    """
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f"✓ Exists, skip: {dest}")
        return
    print(f"→ Downloading {url}")
    tmp = dest.with_suffix(dest.suffix + ".part")
    urlretrieve(url, tmp)
    tmp.replace(dest)
    print(f"   Saved: {dest}")

# Download Yellow Taxi monthly parquet files
yellow_dir = Path("./yellow_2024")
for m in MONTHS_2024:
    fname = f"yellow_tripdata_{m}.parquet"
    download(f"{BASE_URL}/{fname}", yellow_dir / fname)

# Download taxi zone lookup CSV
download(ZONES_URL, Path("./taxi_zone_lookup.csv"))

print("✅ All required TLC files are present.")



In [ ]:
# ============================================================
# NYC Yellow 2024 — Monthly Clean (NO IQR) + Stepwise Drop Report
#   - If output exists and OVERWRITE=False → SKIP (no compute-only)
#   - Only months actually processed appear in the report
# ============================================================

import os, re, glob, shutil
from typing import Dict, Optional, List, Tuple

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, to_timestamp, date_trunc, hour, dayofweek, month,
    unix_timestamp, when, lit, broadcast, concat_ws
)

# -----------------------------
# CONFIG
# -----------------------------
RAW_DIR           = "./yellow_2024"                 # Input: TLC monthly raw parquet folder
CLEAN_DIR_MONTHLY = "./cleaned_data_with_zones"     # Output: per-month cleaned parquet (NO IQR)
ZONE_LOOKUP_CSV   = "./taxi_zone_lookup.csv"        # TLC zone lookup (CSV)
TMP_DIR           = "./_tmp_write"                  # Temp folder for single-file parquet write
OVERWRITE         = False                           # Idempotency control

os.makedirs(CLEAN_DIR_MONTHLY, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# -----------------------------
# SPARK (quiet + lean)
# -----------------------------
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("Yellow-2024-Clean-No-IQR")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "6g")
    .config("spark.driver.maxResultSize", "1g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.files.maxPartitionBytes", str(32*1024*1024))
    .config("spark.local.dir", "./spark_tmp")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark OK:", spark.version)

# -----------------------------
# LOOKUP → broadcast views (once)
# -----------------------------
zones = (
    spark.read.option("header", True).csv(ZONE_LOOKUP_CSV)
    .selectExpr("cast(LocationID as int) as LocationID", "Borough", "Zone", "service_zone")
)

zones_pickup = zones.selectExpr(
    "LocationID as PU_join_id",
    "Borough as pickup_borough",
    "Zone as pickup_zone",
    "service_zone as pickup_service_zone"
)
zones_dropoff = zones.selectExpr(
    "LocationID as DO_join_id",
    "Borough as dropoff_borough",
    "Zone as dropoff_zone",
    "service_zone as dropoff_service_zone"
)

# Broadcast for efficient joins
zones_pickup = broadcast(zones_pickup)
zones_dropoff = broadcast(zones_dropoff)

# -----------------------------
# CONSTANTS / HELPERS
# -----------------------------
KEEP_COLS = [
    "tpep_pickup_datetime","tpep_dropoff_datetime",
    "PULocationID","DOLocationID",
    "fare_amount","tip_amount","total_amount",
    "trip_distance","payment_type","passenger_count"
]

def month_tag(path: str) -> str:
    """
    Extract the 'YYYY-MM' month tag from a given file path.

    Parameters
    ----------
    path : str
        Path to a TLC monthly parquet file (e.g., .../yellow_tripdata_2024-01.parquet).

    Returns
    -------
    str
        Extracted month tag in 'YYYY-MM' format; 'unknown' if no match.
    """
    m = re.search(r"(20\d{2}-\d{2})", path)
    return m.group(1) if m else "unknown"

def write_single_parquet(df: DataFrame, out_file: str):
    """
    Persist a Spark DataFrame as a single snappy-compressed parquet file.

    The function writes to a temporary directory with coalesce(1), then moves the
    single 'part-*.parquet' file to the requested output path to guarantee one file.

    Parameters
    ----------
    df : DataFrame
        Spark DataFrame to write.
    out_file : str
        Destination parquet file path (not a directory).
    """
    tmp_dir = out_file + ".__tmp__"
    shutil.rmtree(tmp_dir, ignore_errors=True)
    (df.coalesce(1)
       .write.mode("overwrite")
       .option("compression", "snappy")
       .parquet(tmp_dir))
    import glob as _glob
    parts = _glob.glob(os.path.join(tmp_dir, "part-*.parquet"))
    if not parts:
        raise RuntimeError(f"No part file in {tmp_dir}")
    os.makedirs(os.path.dirname(out_file), exist_ok=True)
    shutil.move(parts[0], out_file)
    shutil.rmtree(tmp_dir, ignore_errors=True)

def discover_raw_months(raw_dir: str) -> List[Tuple[str, str]]:
    """
    Discover available 2024 monthly raw parquet files in the given directory.

    Parameters
    ----------
    raw_dir : str
        Directory that contains 'yellow_tripdata_2024-*.parquet' files.

    Returns
    -------
    List[Tuple[str, str]]
        Sorted list of (month_tag, file_path) tuples for all matching files.
    """
    paths = sorted(glob.glob(os.path.join(raw_dir, "yellow_tripdata_2024-*.parquet")))
    out = []
    for p in paths:
        tag = month_tag(p)
        if tag != "unknown":
            out.append((tag, p))
    return out

# -----------------------------
# 1) MONTHLY CLEAN (NO IQR) — joins zone names, hard caps only
#     + stepwise counts for drop report
# -----------------------------
def clean_month_no_iqr(
    input_path: str,
    out_path: Optional[str],
    *,
    keep_payment_types=(1, 2),           # 1=Credit, 2=Cash
    hard_caps = {
        "trip_duration_min": (0, 180),   # (0, 3h]  — preserves JFK/long-haul, caps absurd durations
        "trip_distance": (0.1, 60),      # miles    — allows airport runs, removes extreme distances
        "fare_amount": (0, 500),         # dollars  — removes implausibly high fares
        "total_amount": (0, None),       # dollars  — enforce strictly positive totals
    }
) -> Dict[str, object]:
    """
    Clean a single monthly parquet (NO IQR) with domain caps and zone joins,
    track stepwise row counts, and optionally write a single-file parquet output.

    Steps
    -----
    1) Load minimal columns
    2) Core cleaning (valid timestamps, positive counts/fare/total, allowed payments)
    3) Feature engineering (timestamps, HOD/DOW/month, duration, tip %, tag)
    4) Hard caps only (duration, distance, fare, total)
    5) Zone joins (pickup & dropoff) with broadcast lookup
    6) Dependency sanity (total_amount ≥ fare_amount)
    7) De-duplication via composite 'row_key'
    8) Optional write as single parquet file

    Parameters
    ----------
    input_path : str
        Source monthly parquet file path.
    out_path : Optional[str]
        Destination file path for the cleaned single parquet (None = no write).
    keep_payment_types : tuple
        Payment types to retain (1=Credit, 2=Cash).
    hard_caps : dict
        Domain caps for key numeric attributes.

    Returns
    -------
    Dict[str, object]
        Summary with 'tag', 'rows' (stepwise counts), 'drops' (diffs), and 'output_path'.
    """
    tag = month_tag(input_path)
    step = {}

    # 1) Load minimal columns
    df = spark.read.parquet(input_path).select(*KEEP_COLS)
    step["loaded"] = df.count()

    # 2) Core cleaning
    df = (
        df
        .withColumn("pickup_ts",  to_timestamp(col("tpep_pickup_datetime")))
        .withColumn("dropoff_ts", to_timestamp(col("tpep_dropoff_datetime")))
        .filter(col("pickup_ts").isNotNull() & col("dropoff_ts").isNotNull())
        .filter(col("passenger_count").isNotNull() & (col("passenger_count") > 0))
        .filter(col("trip_distance").isNotNull() & (col("trip_distance") > 0))
        .filter(col("fare_amount").isNotNull()   & (col("fare_amount") > 0))
        .filter(col("total_amount").isNotNull()  & (col("total_amount") > 0))
        .filter(col("payment_type").isin(*keep_payment_types))
    )
    step["after_core"] = df.count()

    # 3) Features
    df = (
        df
        .withColumn("pickup_hour", date_trunc("hour", col("pickup_ts")))
        .withColumn("hod", hour(col("pickup_ts")))
        .withColumn("dow", dayofweek(col("pickup_ts")))          # 1=Sun..7=Sat
        .withColumn("month", month(col("pickup_ts")))
        .withColumn("trip_duration_min", (unix_timestamp("dropoff_ts") - unix_timestamp("pickup_ts"))/60.0)
        .withColumn("tip_pct", when(col("fare_amount") > 0, col("tip_amount")/col("fare_amount")).otherwise(0.0))
        .withColumn("year_month", lit(tag))
    )
    step["after_features"] = df.count()

    # 4) Hard caps (domain sanity only, NO IQR)
    if hard_caps.get("trip_duration_min"):
        lo, hi = hard_caps["trip_duration_min"]
        df = df.filter((col("trip_duration_min") > lo) & (col("trip_duration_min") <= hi))
    step["after_cap_duration"] = df.count()

    if hard_caps.get("trip_distance"):
        lo, hi = hard_caps["trip_distance"]
        df = df.filter((col("trip_distance") >= lo) & (col("trip_distance") <= hi))
    step["after_cap_distance"] = df.count()

    if hard_caps.get("fare_amount"):
        lo, hi = hard_caps["fare_amount"]
        cond = col("fare_amount") >= lo
        if hi is not None: cond = cond & (col("fare_amount") <= hi)
        df = df.filter(cond)
    step["after_cap_fare"] = df.count()

    if hard_caps.get("total_amount"):
        lo, hi = hard_caps["total_amount"]
        cond = col("total_amount") >= lo
        if hi is not None: cond = cond & (col("total_amount") <= hi)
        df = df.filter(cond)
    step["after_cap_total"] = df.count()

    # 5) Join zone names (pickup & dropoff)
    df = (
        df.join(zones_pickup, on=col("PULocationID")==col("PU_join_id"), how="left").drop("PU_join_id")
          .join(zones_dropoff, on=col("DOLocationID")==col("DO_join_id"), how="left").drop("DO_join_id")
    )
    # Enforce reference integrity (known zones only)
    df = df.filter(col("pickup_zone").isNotNull() & col("dropoff_zone").isNotNull())
    step["after_join_zones"] = df.count()

    # 6) Dependency sanity: total ≥ fare
    df = df.filter(col("total_amount") >= col("fare_amount"))
    step["after_total_ge_fare"] = df.count()

    # 7) De-duplication
    df = (
        df.withColumn(
            "row_key",
            concat_ws("|",
                col("tpep_pickup_datetime").cast("string"),
                col("tpep_dropoff_datetime").cast("string"),
                col("PULocationID"), col("DOLocationID"),
                col("fare_amount"), col("trip_distance"), col("payment_type")
            )
        )
        .dropDuplicates(["row_key"])
        .drop("row_key")
    )
    step["after_dedup"] = df.count()

    # 8) Write exactly one parquet file for this month
    if out_path:
        write_single_parquet(df, out_path)

    # Build stepwise Δ drops for reporting
    order = [
        "loaded","after_core","after_features",
        "after_cap_duration","after_cap_distance","after_cap_fare","after_cap_total",
        "after_join_zones","after_total_ge_fare","after_dedup"
    ]
    drops = {}
    for i in range(1, len(order)):
        prev, cur = order[i-1], order[i]
        drops[f"drop_{prev}_to_{cur}"] = step[prev] - step[cur]

    summary = {
        "tag": tag,
        "rows": step,
        "drops": drops,
        "output_path": out_path
    }
    print(f"[clean-no-IQR] {tag} rows:", step)
    print(f"[clean-no-IQR] {tag} stepwise drops:", drops)
    return summary

# -----------------------------
# RUN monthly clean (NO IQR) — idempotent
# -----------------------------
raw_months = discover_raw_months(RAW_DIR)
if not raw_months:
    raise FileNotFoundError(f"No raw parquet files found under {RAW_DIR}")

summaries = []
for tag, raw_path in raw_months:
    out_file = os.path.join(CLEAN_DIR_MONTHLY, f"yellow_tripdata_clean_{tag}.parquet")

    if os.path.exists(out_file) and not OVERWRITE:
        # STRICT SKIP: do not recompute counts; preserve prior outputs/prints
        print(f"[skip] {tag} already exists → {out_file}")
        continue

    print(f"\n=== CLEAN (no IQR) — {tag} ===")
    s = clean_month_no_iqr(raw_path, out_path=out_file)
    summaries.append(s)

print("\nMonthly cleaning complete. Folder:", CLEAN_DIR_MONTHLY)

# -----------------------------
# 2) WRITE stepwise report CSV (only if some months processed)
# -----------------------------
import pandas as pd

if len(summaries) == 0:
    print("No months newly processed — skipping cleaning report CSV.")
else:
    rows = []
    for s in summaries:
        tag = s["tag"]; step = s["rows"]; drops = s["drops"]
        rows.append({
            "month": tag,
            # absolute counts
            **{k: step.get(k, None) for k in [
                "loaded","after_core","after_features",
                "after_cap_duration","after_cap_distance","after_cap_fare","after_cap_total",
                "after_join_zones","after_total_ge_fare","after_dedup"
            ]},
            # drops between steps
            **drops
        })

    df_report = pd.DataFrame(rows).sort_values("month")
    out_csv = "cleaning_step_drops.csv"
    df_report.to_csv(out_csv, index=False)
    print(f"Saved stepwise cleaning report → {out_csv}")


In [ ]:
import pandas as pd

# Read raw events CSV
events = pd.read_csv("Parks_Special_Events.csv")

# Parse datetime
events['Date and Time'] = pd.to_datetime(events['Date and Time'], errors='coerce')

# Extract date
events['EventDate'] = events['Date and Time'].dt.date

# Keep only 2024 data
events_2024 = events[events['Date and Time'].dt.year == 2024].copy()

# Select useful columns
events_2024 = events_2024[[
    'EventDate',
    'Borough',
    'Location',
    'Event Name',
    'Category',
    'Attendance'
]]

# Convert Attendance to integer (drop .0)
events_2024['Attendance'] = pd.to_numeric(events_2024['Attendance'], errors='coerce').astype('Int64')

# Save to cleaned files
events_2024.to_csv("nyc_events_2024.csv", index=False)


In [ ]:
# ============================================================
# NYC Yellow Taxi 2024 — ONE-STOP ANALYSIS & MODELLING SCRIPT
# ============================================================
# What this does:
# - Loads your cleaned parquet (prefers ./cleaned_data_with_zones_global, else ./cleaned_data_with_zones)
# - Ingests external CSVs: nyc_all_holidays_2024.csv and (optional) nyc_events_2024.csv
# - Reproduces ALL figures (hour/week heatmaps, wkday vs wkend, holiday/event effects,
#   top zones, geospatial maps, monthly trends, fare/tip curves, revenue/trip by borough, etc.)
# - Trains & evaluates LR vs RF on hourly (date×hour×borough) trip counts; saves metrics + importances
# - Writes outputs to a NEW timestamped folder so previous figures aren’t overwritten.
# ============================================================

import os, sys, zipfile, urllib.request, subprocess, warnings, tempfile
from datetime import datetime
warnings.filterwarnings("ignore")

# ---------- Output root (FIXED FOLDER NAME) ----------
# Creates a stable results folder structure (figures, tables, logs).
OUT_ROOT = os.path.abspath("./nyc_yellow_2024_run")
FIG_DIR = os.path.join(OUT_ROOT, "figures")
TAB_DIR = os.path.join(OUT_ROOT, "tables")
LOG_DIR = os.path.join(OUT_ROOT, "logs")
for d in [OUT_ROOT, FIG_DIR, TAB_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

def log(msg):
    """Append a timestamped line to console and run log."""
    s = f"[NYC2024] {msg}"
    print(s)
    with open(os.path.join(LOG_DIR, "run.log"), "a") as f:
        f.write(s + "\n")

# ---------- TMP/Spark temp dirs (avoid /tmp low space) ----------
# Direct large Spark temps to a controlled project directory.
TMP_SAFE = os.path.abspath(os.path.join(OUT_ROOT, "_spark_tmp"))
os.makedirs(TMP_SAFE, exist_ok=True)
for k in ["TMPDIR", "TEMP", "TMP", "SPARK_LOCAL_DIRS"]:
    os.environ[k] = TMP_SAFE
tempfile.tempdir = TMP_SAFE

# ---------- Imports ----------
# Core scientific stack and plotting. Seaborn used for high-level aesthetics.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm
import seaborn as sns
sns.set(style="whitegrid", font_scale=1.15)

# Guardrail for expected pandas version.
if int(pd.__version__.split(".")[0]) < 2:
    raise RuntimeError(f"Pandas {pd.__version__} detected. Please install pandas>=2.0.")

# Spark + helpers
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    to_timestamp, hour, dayofweek, month, to_date,
    sum as Fsum, avg as Favg, expr, col, when
)
from py4j.protocol import Py4JJavaError

# ---------- Spark Session ----------
# Lightweight local Spark with adaptive execution and tuned shuffle partitions.
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("NYC_Yellow_2024_OneStop")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.local.dir", TMP_SAFE)
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.files.maxPartitionBytes", str(64*1024*1024))
    .config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
log("Spark started.")

# ---------- Helpers ----------
def _ensure_cols(df):
    """
    Keep required columns, derive time features (hod, dow, month), and compute tip_pct.
    Ensures a consistent minimally-required schema for all downstream steps.
    """
    needed = {
        "tpep_pickup_datetime","PULocationID","DOLocationID",
        "total_amount","fare_amount","tip_amount",
        "trip_distance","trip_duration_min",
        "pickup_borough","pickup_zone"
    }
    keep = [c for c in df.columns if c in needed]
    df = df.select(*keep)
    df = (
        df.withColumn("pickup_ts", to_timestamp("tpep_pickup_datetime"))
          .withColumn("hod",  hour("pickup_ts"))
          .withColumn("dow",  dayofweek("pickup_ts"))   # 1=Sun..7=Sat
          .withColumn("month", month("pickup_ts"))
    )
    if "tip_pct" not in df.columns and "fare_amount" in df.columns and "tip_amount" in df.columns:
        df = df.withColumn(
            "tip_pct",
            when(col("fare_amount") > 0, col("tip_amount")/col("fare_amount")).otherwise(0.0)
        )
    return df

def run_or_sample(spark_action, sample_fraction=0.25, what="operation"):
    """
    Execute a computation on the full dataset; if a disk-space failure occurs,
    fall back to a reproducible sample and log the downgrade.
    """
    try:
        return spark_action(taxi_full)
    except Py4JJavaError as e:
        msg = str(e.java_exception) if hasattr(e, "java_exception") else str(e)
        if "No space left on device" in msg:
            log(f"[LowDisk] '{what}' failed → using {int(sample_fraction*100)}% sample.")
            taxi_small = taxi_full.sample(withReplacement=False, fraction=sample_fraction, seed=42)
            return spark_action(taxi_small)
        raise

def _ensure_geopandas():
    """Import geopandas stack on demand; install if missing."""
    try:
        import geopandas as gpd  # noqa: F401
    except Exception:
        log("[Geo] Installing geopandas stack...")
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "geopandas", "shapely", "fiona", "pyproj", "pyogrio"])
    import geopandas as gpd
    return gpd

def _load_tlc_zones_from_zip():
    """
    Fetch TLC taxi zone shapefile (if absent), load as GeoDataFrame, and normalize the LocationID column.
    """
    shp_dir = os.path.join(OUT_ROOT, "taxi_zones_shp")
    os.makedirs(shp_dir, exist_ok=True)
    shp = next((os.path.join(shp_dir, f) for f in os.listdir(shp_dir) if f.lower().endswith(".shp")), None)
    if shp is None:
        url_zip = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"
        zip_path = os.path.join(shp_dir, "taxi_zones.zip")
        try:
            log("[Geo] Downloading TLC taxi_zones.zip ...")
            urllib.request.urlretrieve(url_zip, zip_path)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(shp_dir)
            shp = next((os.path.join(shp_dir, f) for f in os.listdir(shp_dir) if f.lower().endswith(".shp")), None)
        except Exception as e:
            raise RuntimeError("Download blocked. Manually unzip TLC taxi_zones.zip into the run folder.") from e
    if shp is None:
        raise FileNotFoundError("No .shp in taxi_zones_shp/.")
    gpd = _ensure_geopandas()
    gdf = gpd.read_file(shp)
    gdf = gdf.loc[:, ~pd.Index(gdf.columns).duplicated()].copy()
    for c in ["LocationID","location_id","Location_Id","loc_id","LOC_ID"]:
        if c in gdf.columns:
            gdf["loc_id"] = gdf[c].astype(int)
            break
    if "loc_id" not in gdf.columns:
        raise ValueError("Zones file missing LocationID-like column.")
    return gdf

# ---------- Load cleaned parquet (NO GLOBAL IQR, single clean folder) ----------
# Loads all monthly cleaned parquet files produced by the cleaning pipeline.
CLEAN_MONTHLY = "./cleaned_data_with_zones"

if not os.path.isdir(CLEAN_MONTHLY):
    raise FileNotFoundError(
        "Folder ./cleaned_data_with_zones not found. "
        "Run the cleaning pipeline (no IQR) to produce monthly cleaned parquet first."
    )

paths = [
    os.path.join(CLEAN_MONTHLY, f)
    for f in sorted(os.listdir(CLEAN_MONTHLY))
    if f.endswith(".parquet")
]

if not paths:
    raise FileNotFoundError(
        "No cleaned parquet found in ./cleaned_data_with_zones. "
        "Run the cleaning pipeline (no IQR) to generate them."
    )

taxi_full = _ensure_cols(spark.read.parquet(*paths))
log(f"Loaded cleaned parquet (no IQR): {len(paths)} files from {CLEAN_MONTHLY}.")

# ---------- External data ----------
# Holidays
# Normalizes holiday dates and keeps a compact reference table.
hol = pd.read_csv("nyc_all_holidays_2024.csv")
if "Date" in hol.columns: hol["date"] = pd.to_datetime(hol["Date"]).dt.normalize()
elif "date" in hol.columns: hol["date"] = pd.to_datetime(hol["date"]).dt.normalize()
else: raise ValueError("nyc_all_holidays_2024.csv must have 'Date' or 'date'.")
hol = hol[["date","Holiday"]].drop_duplicates()

# Events (optional)
# Prepare both daily event counts and borough-level event-day flags if available.
events_path = "nyc_events_2024.csv"
ev_day, ev_boro, ev_raw = None, None, None
if os.path.exists(events_path):
    ev_raw = pd.read_csv(events_path)
    date_col = next((c for c in ["EventDate","date","Date"] if c in ev_raw.columns), None)
    if date_col:
        ev_raw["date"] = pd.to_datetime(ev_raw[date_col], errors="coerce").dt.normalize()
        ev_day = ev_raw.groupby("date").size().reset_index(name="event_count")
        boro_col = "Borough" if "Borough" in ev_raw.columns else ("borough" if "borough" in ev_raw.columns else None)
        if boro_col:
            ev_boro = (ev_raw.groupby(["date", boro_col]).size()
                        .reset_index(name="event_count_boro")
                        .rename(columns={boro_col:"pickup_borough"}))

# ---------- Figure helpers ----------
def savefig(name):
    """Save current figure into figures/ with tight layout and log the artifact path."""
    path = os.path.join(FIG_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.show()
    log(f"Saved figure: {path}")
    return path

def savetab(df, name):
    """Save DataFrame as CSV into tables/ and log the artifact path."""
    path = os.path.join(TAB_DIR, name)
    df.to_csv(path, index=False)
    log(f"Saved table: {path}")
    return path

# ============================================================
# 1) CORE FIGURES
# ============================================================
def fig_hour_of_week(metric="revenue"):
    """Heatmap of hour-of-week for revenue or trips."""
    colname = "revenue" if metric=="revenue" else "trips"
    def action(df):
        agg = Fsum("total_amount").alias(colname) if colname=="revenue" else Fsum(expr("1")).alias(colname)
        return df.groupBy("dow","hod").agg(agg).toPandas()
    pdf = run_or_sample(action, what=f"hour-of-week {metric}")
    dow_map = {1:"Sun",2:"Mon",3:"Tue",4:"Wed",5:"Thu",6:"Fri",7:"Sat"}
    pdf["dow_name"] = pdf["dow"].map(dow_map)
    order = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
    piv = pdf.pivot(index="dow_name", columns="hod", values=colname).reindex(order)
    plt.figure(figsize=(16,4))
    sns.heatmap(piv, cmap="magma", linewidths=.3, cbar_kws={"label":"Revenue ($)" if metric=="revenue" else "Trips"})
    plt.title(f"Hour-of-Week Heatmap — {metric.capitalize()} (2024)")
    plt.xlabel("Hour of Day"); plt.ylabel("Day of Week")
    savefig(f"01_heatmap_hourweek_{metric}.png")

def fig_weekday_weekend():
    """Hourly revenue curves for weekdays vs weekends."""
    pdf = run_or_sample(
        lambda df: df.groupBy("hod","dow")
                    .agg(Fsum("total_amount").alias("revenue"))
                    .withColumn("is_weekend", col("dow").isin([1,7]).cast("int"))
                    .toPandas(),
        what="weekday vs weekend revenue"
    )
    pdf["daytype"] = pdf["is_weekend"].map({0:"Weekday",1:"Weekend"})
    curve = pdf.groupby(["daytype","hod"], as_index=False)["revenue"].sum()
    plt.figure(figsize=(12,5))
    sns.lineplot(data=curve, x="hod", y="revenue", hue="daytype", marker="o")
    plt.title("Hourly Revenue — Weekday vs Weekend (2024)")
    plt.xlabel("Hour of Day"); plt.ylabel("Revenue ($)")
    savefig("02_weekday_weekend_revenue.png")

def fig_holiday_nonholiday():
    """Mean hourly revenue on holidays vs non-holidays."""
    pdf = run_or_sample(
        lambda df: df.groupBy(to_date("pickup_ts").alias("date"), "hod")
                    .agg(Fsum("total_amount").alias("revenue")).toPandas(),
        what="holiday hourly revenue"
    )
    pdf["date"] = pd.to_datetime(pdf["date"]).dt.normalize()
    pdf = pdf.merge(hol, on="date", how="left")
    pdf["is_holiday"] = pdf["Holiday"].notna().map({True:"Holiday", False:"Non-Holiday"})
    curve = pdf.groupby(["is_holiday","hod"], as_index=False)["revenue"].mean()
    plt.figure(figsize=(12,5))
    sns.lineplot(data=curve, x="hod", y="revenue", hue="is_holiday", marker="o")
    plt.title("Hourly Revenue — Holiday vs Non-Holiday (2024)")
    plt.xlabel("Hour of Day"); plt.ylabel("Mean Revenue ($)")
    savefig("03_holiday_nonholiday.png")

def fig_event_borough():
    """Mean hourly revenue in event-borough vs non-event-borough days (if events have boroughs)."""
    if ev_boro is None or ev_boro.empty:
        log("[Info] No borough column in events; skipping Event vs Non-Event Borough figure.")
        return
    hb = run_or_sample(
        lambda df: df.groupBy(to_date("pickup_ts").alias("date"), "hod", "pickup_borough")
                    .agg(Fsum("total_amount").alias("revenue")).toPandas(),
        what="event borough hourly revenue"
    )
    hb["date"] = pd.to_datetime(hb["date"]).dt.normalize()
    ev_b = ev_boro.copy()
    ev_b["date"] = pd.to_datetime(ev_b["date"]).dt.normalize()
    hb = hb.merge(ev_b, on=["date","pickup_borough"], how="left")
    hb["is_event_borough"] = (hb["event_count_boro"].fillna(0) > 0).map({True:"Event Borough",False:"Non-Event Borough"})
    curve = hb.groupby(["is_event_borough","hod"], as_index=False)["revenue"].mean()
    plt.figure(figsize=(12,5))
    sns.lineplot(data=curve, x="hod", y="revenue", hue="is_event_borough", marker="o")
    plt.title("Hourly Revenue — Event vs Non-Event Borough (2024)")
    plt.xlabel("Hour of Day"); plt.ylabel("Mean Revenue ($)")
    savefig("04_event_borough.png")

def fig_top_pickups(n=15):
    """Top pickup zones by count."""
    pdf = run_or_sample(lambda df: df.groupBy("PULocationID")
                                   .agg(Fsum(expr("1")).alias("pickups")).toPandas(),
                        what="top zones pickups")
    zl = pd.read_csv("taxi_zone_lookup.csv"); zl["LocationID"]=zl["LocationID"].astype(int)
    pdf = pdf.merge(zl, left_on="PULocationID", right_on="LocationID", how="left")
    pdf = pdf[~pdf["Borough"].isin(["Unknown","N/A"]) & pdf["Borough"].notna()]
    pdf = pdf.nlargest(n, "pickups")
    plt.figure(figsize=(12,7))
    sns.barplot(data=pdf, y="Zone", x="pickups", hue="Borough", dodge=False)
    plt.title(f"Top {n} Pickup Zones (2024)"); plt.xlabel("Pickups"); plt.ylabel("TLC Zone")
    plt.legend(title="Borough", bbox_to_anchor=(1.02, 1), loc="upper left")
    savefig("05_top_pickups.png")

def fig_top_revenue(n=15):
    """Top pickup zones by revenue."""
    pdf = run_or_sample(lambda df: df.groupBy("PULocationID")
                                   .agg(Fsum("total_amount").alias("revenue")).toPandas(),
                        what="top zones revenue")
    zl = pd.read_csv("taxi_zone_lookup.csv"); zl["LocationID"]=zl["LocationID"].astype(int)
    pdf = pdf.merge(zl, left_on="PULocationID", right_on="LocationID", how="left")
    pdf = pdf[~pdf["Borough"].isin(["Unknown","N/A"]) & pdf["Borough"].notna()]
    pdf = pdf.nlargest(n, "revenue")
    plt.figure(figsize=(12,7))
    sns.barplot(data=pdf, y="Zone", x="revenue", hue="Borough", dodge=False)
    plt.title(f"Top {n} Zones by Revenue (2024)"); plt.xlabel("Revenue ($)"); plt.ylabel("TLC Zone")
    plt.legend(title="Borough", bbox_to_anchor=(1.02, 1), loc="upper left")
    savefig("06_top_revenue.png")

def fig_geo_maps():
    """Choropleths: revenue and average tip% by pickup zone (with capped tip scale)."""
    lookup_csv = "taxi_zone_lookup.csv"
    if not os.path.exists(lookup_csv):
        log("taxi_zone_lookup.csv not found; skipping maps.")
        return
    rev = run_or_sample(lambda df: df.groupBy("PULocationID")
                                      .agg(Fsum("total_amount").alias("revenue"),
                                           Favg("tip_pct").alias("avg_tip_pct")).toPandas(),
                        what="geo revenue & tips")
    rev = rev.merge(pd.read_csv(lookup_csv).assign(LocationID=lambda d: d["LocationID"].astype(int)),
                    left_on="PULocationID", right_on="LocationID", how="left")
    try:
        gpd = _ensure_geopandas(); gzones = _load_tlc_zones_from_zip()
        g = gzones.merge(rev.rename(columns={"PULocationID":"loc_id"}), on="loc_id", how="left")
        g["revenue"] = g["revenue"].fillna(0)
        g["avg_tip_pct"] = g["avg_tip_pct"].astype(float)
        fig, ax = plt.subplots(1,2, figsize=(16,8))
        for a in ax: a.axis("off")
        g.plot(column="revenue", cmap="magma", ax=ax[0], legend=True, legend_kwds={"label":"Revenue ($)"})
        ax[0].set_title("Revenue by Pickup Zone (2024)")
        g["avg_tip_pct_capped"] = g["avg_tip_pct"].clip(lower=0, upper=np.nanquantile(g["avg_tip_pct"], 0.99))
        g.plot(column="avg_tip_pct_capped", cmap="viridis", ax=ax[1], legend=True, legend_kwds={"label":"Avg Tip %"})
        ax[1].set_title("Average Tip % by Pickup Zone (2024)")
        savefig("07_geo_revenue_tip.png")
    except Exception as e:
        log(f"[Geo] Map rendering skipped: {e}")
        rev.rename(columns={"PULocationID":"loc_id"})[["loc_id","revenue","avg_tip_pct"]].to_csv(
            os.path.join(TAB_DIR,"revenue_tip_by_zone.csv"), index=False
        )
        log("Saved fallback tables: tables/revenue_tip_by_zone.csv")

def fig_fare_tip_by_hour():
    """Average fare and average tip% by hour-of-day."""
    pdf = run_or_sample(lambda df: df.groupBy("hod")
                    .agg(Favg("fare_amount").alias("avg_fare"),
                         Favg("tip_pct").alias("avg_tip")).toPandas(),
                    what="fare/tip by hour")
    fig, ax = plt.subplots(1,2, figsize=(14,5))
    sns.lineplot(data=pdf, x="hod", y="avg_fare", marker="o", ax=ax[0])
    ax[0].set_title("Average Fare by Hour (2024)"); ax[0].set_xlabel("Hour"); ax[0].set_ylabel("Average Fare ($)")
    sns.lineplot(data=pdf, x="hod", y="avg_tip", marker="o", ax=ax[1])
    ax[1].set_title("Average Tip % by Hour (2024)"); ax[1].set_xlabel("Hour"); ax[1].set_ylabel("Average Tip (%)")
    savefig("09_fare_tip_hour.png")

def fig_revenue_per_trip_by_borough():
    """Revenue-per-trip by pickup borough."""
    pdf = run_or_sample(lambda df: df.groupBy("pickup_borough")
                    .agg((Fsum("total_amount")/Fsum(expr("1"))).alias("rev_per_trip")).toPandas(),
                    what="rev per trip by borough")
    pdf = pdf.sort_values("rev_per_trip", ascending=False)
    plt.figure(figsize=(8,5))
    sns.barplot(data=pdf, x="pickup_borough", y="rev_per_trip")
    plt.title("Revenue per Trip by Borough (2024)"); plt.ylabel("Revenue per Trip ($)"); plt.xlabel("")
    savefig("10_revenue_trip_borough.png")
    savetab(pdf, "revenue_per_trip_borough.csv")

# Extras (events + monthly + simple tip curve + pickups/dropoffs maps)
def fig_events_by_borough_and_top_venues():
    """Optional diagnostics: events count by borough and top venues (if event data available)."""
    if ev_raw is None or ev_raw.empty:
        return
    if "Borough" in ev_raw.columns or "borough" in ev_raw.columns:
        boro_col = "Borough" if "Borough" in ev_raw.columns else "borough"
        e_b = ev_raw.groupby(boro_col).size().reset_index(name="events")
        plt.figure(figsize=(10,5))
        sns.barplot(data=e_b, x=boro_col, y="events")
        plt.title("Events by Borough (2024)"); plt.ylabel("# Events"); plt.xlabel("")
        savefig("events_by_borough.png")
        savetab(e_b, "events_by_borough.csv")
    venue_col = next((c for c in ["Venue","Location","EventLocation","FacilityName"] if c in ev_raw.columns), None)
    if venue_col:
        e_v = (ev_raw.groupby(venue_col).size().reset_index(name="events")
               .sort_values("events", ascending=False).head(15))
        plt.figure(figsize=(12,7))
        sns.barplot(data=e_v, y=venue_col, x="events")
        plt.title("Top Event Venues (2024)"); plt.xlabel("# Events"); plt.ylabel("")
        savefig("events_top_venues.png")
        savetab(e_v, "events_top_venues.csv")

def fig_monthly_trends():
    """Monthly trips and revenue bars for 2024."""
    pdf = run_or_sample(lambda df: df.groupBy("month")
                    .agg(Fsum(expr("1")).alias("trips"),
                         Fsum("total_amount").alias("revenue")).toPandas(),
                    what="monthly trends")
    pdf = pdf.sort_values("month")
    fig, ax = plt.subplots(1,2, figsize=(14,4))
    sns.barplot(data=pdf, x="month", y="trips", ax=ax[0])
    ax[0].set_title("Trips by Month (2024)"); ax[0].set_xlabel("Month"); ax[0].set_ylabel("Trips")
    sns.barplot(data=pdf, x="month", y="revenue", ax=ax[1])
    ax[1].set_title("Revenue by Month (2024)"); ax[1].set_xlabel("Month"); ax[1].set_ylabel("$")
    savefig("monthly_trends_trips_revenue.png")
    savetab(pdf, "monthly_trips_revenue.csv")

def fig_tip_pct_by_hour_simple():
    """Simple tip% vs hour diagnostic."""
    pdf = run_or_sample(lambda df: df.groupBy("hod").agg(Favg("tip_pct").alias("tip_pct")).toPandas(),
                        what="tip pct by hour")
    plt.figure(figsize=(12,4))
    sns.lineplot(data=pdf, x="hod", y="tip_pct", marker="o")
    plt.title("Average Tip % by Hour"); plt.xlabel("Hour"); plt.ylabel("Tip %")
    savefig("tip_pct_by_hour.png")

def fig_pickups_dropoffs_maps():
    """Geospatial pickup and dropoff intensity maps (log-normalized) by zone."""
    lookup_csv = "taxi_zone_lookup.csv"
    if not os.path.exists(lookup_csv):
        log("taxi_zone_lookup.csv not found; skipping pickups/dropoffs maps.")
        return
    pu = run_or_sample(lambda df: df.groupBy("PULocationID").agg(Fsum(expr("1")).alias("pickups")).toPandas(),
                       what="geo pickups")
    do = run_or_sample(lambda df: df.groupBy("DOLocationID").agg(Fsum(expr("1")).alias("dropoffs")).toPandas(),
                       what="geo dropoffs")
    savetab(pu, "pickups_by_zone_2024.csv")
    savetab(do, "dropoffs_by_zone_2024.csv")
    try:
        gpd = _ensure_geopandas(); gzones = _load_tlc_zones_from_zip()
        zl = pd.read_csv(lookup_csv); zl["LocationID"] = zl["LocationID"].astype(int)
        pu_m = gzones.merge(pu.rename(columns={"PULocationID":"loc_id"}), on="loc_id", how="left").fillna(0)
        do_m = gzones.merge(do.rename(columns={"DOLocationID":"loc_id"}), on="loc_id", how="left").fillna(0)
        fig, axes = plt.subplots(1,2, figsize=(16,8)); [ax.axis("off") for ax in axes]
        vmin = 1.0; vmax = max(vmin, pu_m["pickups"].max() or 1.0)
        pu_m.plot(column="pickups", cmap="magma", ax=axes[0],
                  norm=LogNorm(vmin=vmin, vmax=vmax), edgecolor="none")
        axes[0].set_title("Yellow 2024 — Total Pick-ups by TLC Zone")
        vmin2 = 1.0; vmax2 = max(vmin2, do_m["dropoffs"].max() or 1.0)
        do_m.plot(column="dropoffs", cmap="magma", ax=axes[1],
                  norm=LogNorm(vmin=vmin2, vmax=vmax2), edgecolor="none")
        axes[1].set_title("Yellow 2024 — Total Drop-offs by TLC Zone")
        savefig("geo_pickups_dropoffs_2024.png")
    except Exception as e:
        log(f"[Geo] pickups/dropoffs map skipped: {e}")

# ============================================================
# 2) MODELLING — Predict hourly borough×date REVENUE
#      • Train = first 10 months (i.e., all but last 2 months)
#      • Test  = last 2 months
#      • Models: Linear Regression, Random Forest
#      • Saves: metrics CSV, parity plot, residuals-over-time (RF), RF importances
# ============================================================
# ============================================================
# 2) MODELLING — Predict hourly borough×date REVENUE
#      • Data locked to year 2024
#      • Train = Jan–Oct, Test = Nov–Dec
#      • Models: Linear Regression, Random Forest
# ============================================================
def modelling_revenue():
    """
    Build hourly revenue per (date × hour × borough), enrich with holiday/event flags,
    split Train (Jan–Oct) vs Test (Nov–Dec), fit Linear Regression and Random Forest,
    evaluate (RMSE/MAE/R²), persist metrics and diagnostics, and return metrics DataFrame.
    """
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    from sklearn.linear_model import LinearRegression
    from sklearn.ensemble import RandomForestRegressor
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    # ---------- Build modelling frame (hourly revenue per borough×date) ----------
    def build_frame():
        """Aggregate hourly revenue by date, hour, day-of-week and borough; merge calendar flags."""
        def action(df):
            return (df.groupBy(to_date("pickup_ts").alias("date"), "hod", "dow", "pickup_borough")
                      .agg(Fsum("total_amount").alias("revenue")).toPandas())
        hbd = run_or_sample(action, what="modelling base")
        hbd["date"] = pd.to_datetime(hbd["date"]).dt.normalize()

        # limit strictly to 2024
        hbd = hbd[(hbd["date"] >= pd.Timestamp("2024-01-01")) &
                  (hbd["date"] <= pd.Timestamp("2024-12-31"))].copy()

        # calendar/enrichment
        hbd = hbd.merge(hol, on="date", how="left")
        hbd["is_holiday"] = hbd["Holiday"].notna().astype(int)

        if 'ev_day' in globals() and ev_day is not None:
            tmp = ev_day.copy()
            tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce").dt.normalize()
            hbd = hbd.merge(tmp, on="date", how="left")
            hbd["event_count"] = hbd["event_count"].fillna(0).astype(int)
            hbd["is_event_day"] = (hbd["event_count"] > 0).astype(int)
        else:
            hbd["is_event_day"] = 0

        hbd["month"] = hbd["date"].dt.month.astype(int)
        hbd["ym"] = hbd["date"].dt.to_period("M")
        return hbd

    hbd = build_frame().sort_values(["date","pickup_borough","hod"])

    # quick sanity log
    ym_counts = hbd["ym"].value_counts().sort_index()
    print("[Model] 2024 month counts:\n", ym_counts.to_string())

    # ---------- Fixed time split: Jan–Oct train, Nov–Dec test ----------
    train_mask = hbd["month"].between(1, 10)   # Jan..Oct
    test_mask  = hbd["month"].between(11, 12)  # Nov..Dec

    X = hbd[["hod","dow","month","is_holiday","is_event_day"]].astype(int)
    X = pd.concat([X, pd.get_dummies(hbd["pickup_borough"], prefix="boro", drop_first=True)], axis=1)
    y = hbd["revenue"].astype(float)

    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]

    print(f"[Model] Train months: {sorted(hbd.loc[train_mask, 'ym'].unique())} "
          f"(n={len(y_train):,})")
    print(f"[Model] Test months : {sorted(hbd.loc[test_mask,  'ym'].unique())} "
          f"(n={len(y_test):,})")

    if len(y_test) == 0:
        raise RuntimeError("Test set is empty — check that your cleaned data actually includes Nov–Dec 2024.")

    # ---------- Fit models ----------
    results = []

    def eval_and_log(model, name):
        """Fit, predict test, compute RMSE/MAE/R², collect metrics, and return model/preds."""
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
        mae  = float(mean_absolute_error(y_test, pred))
        r2   = float(r2_score(y_test, pred))
        results.append({"model": name, "RMSE": rmse, "MAE": mae, "R2": r2})
        return model, pred

    lr, pred_lr = eval_and_log(LinearRegression(), "Linear Regression")
    rf, pred_rf = eval_and_log(
        RandomForestRegressor(n_estimators=400, max_depth=16, min_samples_leaf=2,
                              random_state=42, n_jobs=-1),
        "Random Forest"
    )

    # ---------- Save metrics ----------
    met = pd.DataFrame(results)
    savetab(met, "model_metrics.csv")

    # ---------- Diagnostic plots (test set) ----------
    # 1) Parity (y_true vs y_pred) — both models
    plt.figure(figsize=(7,7))
    plt.scatter(y_test, pred_lr, s=10, alpha=0.4, label="Linear Reg")
    plt.scatter(y_test, pred_rf, s=10, alpha=0.4, label="Random Forest")
    lims = [0, max(y_test.max(), pred_rf.max(), pred_lr.max())]
    plt.plot(lims, lims, linestyle="--", linewidth=1)
    plt.xlabel("Actual revenue (test)"); plt.ylabel("Predicted revenue")
    plt.title("Model parity (test)")
    plt.legend()
    savefig("model_parity_test.png")

    # 2) Residuals over (date×hour×borough) — RF
    test_idx = hbd.index[test_mask]
    rf_resid = y_test.values - pred_rf
    # Put on a sequential index for plotting
    plt.figure(figsize=(12,4))
    plt.plot(rf_resid, linewidth=0.8)
    plt.axhline(0, color="black", linewidth=0.8)
    plt.title("Random Forest residuals (test set order)")
    plt.ylabel("Actual − Predicted")
    plt.xlabel("Test set rows (Jan–Oct train • Nov–Dec test)")
    savefig("model_residuals_over_time_rf.png")

    # 3) RF Feature importance
    importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(10,6))
    importances.head(15).plot(kind="barh")
    plt.gca().invert_yaxis()
    plt.title("Random Forest — Top Feature Importances")
    plt.xlabel("Importance")
    savefig("rf_feature_importance.png")

    # ---------- Human-readable summary ----------
    with open(os.path.join(OUT_ROOT, "metrics_summary.txt"), "w") as f:
        for row in results:
            f.write(f"{row['model']} | RMSE={row['RMSE']:.2f} | MAE={row['MAE']:.2f} | R^2={row['R2']:.3f}\n")

    return met



# ============================================================
# 3) RUN EVERYTHING
# ============================================================
log("Generating figures...")
fig_hour_of_week("revenue")
fig_hour_of_week("trips")
fig_weekday_weekend()
fig_holiday_nonholiday()
fig_event_borough()
fig_top_pickups()
fig_top_revenue()
fig_geo_maps()
fig_pickups_dropoffs_maps()
fig_fare_tip_by_hour()
fig_revenue_per_trip_by_borough()
fig_events_by_borough_and_top_venues()
fig_monthly_trends()
fig_tip_pct_by_hour_simple()

log("Modelling...")
met = modelling_revenue()

log("All done. See folder: " + OUT_ROOT)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Pull full data from Spark to Pandas (no sampling)
pdf = taxi_full.select("total_amount").toPandas()
pdf = pdf.dropna()
pdf = pdf[pdf["total_amount"] > 0]

# --- Single Histogram (full range, log y-axis) ---
plt.figure(figsize=(10,5))
sns.histplot(pdf["total_amount"], bins=200, kde=False, color="steelblue")
plt.xlim(0, 500)
plt.yscale("log")
plt.xlabel("Total Amount ($)")
plt.ylabel("Log(Number of Trips)")
plt.title("Distribution of Total Revenue per Trip")
plt.tight_layout()
plt.savefig("nyc_yellow_2024_run/figures/total_amount_distribution.png", dpi=200)
plt.show()
